In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# Data: ToTensor scales to [0, 1]; the model flattens the images itself
train_data = datasets.MNIST("mnist_data", train=True, transform=ToTensor(), download=True)
test_data = datasets.MNIST("mnist_data", train=False, transform=ToTensor())
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

In [3]:
# Model: images stay as (N, 1, 28, 28), so there is no Flatten at the start
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),   # -> (16, 28, 28)
    nn.ReLU(),
    nn.MaxPool2d(2),                              # -> (16, 14, 14)
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # -> (32, 14, 14)
    nn.ReLU(),
    nn.MaxPool2d(2),                              # -> (32, 7, 7)
    nn.Flatten(),                                 # -> 1568
    nn.Linear(32 * 7 * 7, 10),                    # logits
)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [4]:
# Training
for epoch in range(5):
    for X, y in train_loader:
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch}, last batch loss = {loss.item():.4f}")

epoch 0, last batch loss = 0.0286
epoch 1, last batch loss = 0.0416
epoch 2, last batch loss = 0.0556
epoch 3, last batch loss = 0.0043
epoch 4, last batch loss = 0.0248


In [5]:
# Evaluation
model.eval()  # Switches off Dropout and Batch Normalization, if used
correct = 0
with torch.no_grad():
    for X, y in test_loader:
        correct += (model(X).argmax(dim=1) == y).sum().item()
print(f"Accuracy = {correct / len(test_data):.4f}")

Accuracy = 0.9893
